# Metallicity-dependent MLR test plot

Use the cells below to enter your fitted parameters and visualize the MLR vs. the isochrone at multiple metallicities.


In [2]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Add the package to Python path
def _find_repo_root(start: Path) -> Path:
    for path in [start, *start.parents]:
        if (path / "bayesian-binary-masses" / "src").exists():
            return path
    return start

repo_root = _find_repo_root(Path.cwd().resolve())
sys.path.insert(0, str(repo_root / "bayesian-binary-masses" / "src"))

from binary_masses.differencepoly_feh import (
    DifferencePolyFehMassAbsgModel,
    IsochroneMassSurfaceModel,
)

iso_data_dir = repo_root / "bayesian-binary-masses" / "data" / "interpolated_mass_data"
iso_surface = IsochroneMassSurfaceModel.from_interpolated_mass_data(
    str(iso_data_dir),
    mass_min=0.05,
)


FileNotFoundError: Could not find interpolated mass grid directory. Pass `data_dir=...` (expected to contain `gmag_grid.npy`).

In [ ]:
# -------------------------
# User inputs
# -------------------------
order = 3
absg_min, absg_max = 3.5, 14.0
mass_min, mass_max = 0.05, 2.0
feh_min, feh_max = -1.0, 0.6

# Choose the metallicity values to compare
feh_values = [-1.0, -0.5, 0.0, 0.3, 0.6]

# Option A: single set of coefficients (no uncertainty band)
# a_coeffs and b_coeffs should each have length order+1
a_coeffs = [0.0, 0.0, 0.0, 0.0]
b_coeffs = [0.0, 0.0, 0.0, 0.0]
params = np.array(a_coeffs + b_coeffs, dtype=float)

# Option B: posterior samples (uncertainty band)
# params_samples shape: (n_samples, 2*(order+1))
# params_samples = np.loadtxt("path/to/mcmc_samples.txt")
params_samples = None


In [ ]:
def plot_mlr_multi_feh(
    *,
    params,
    params_samples,
    feh_values,
    absg_min,
    absg_max,
    mass_min,
    mass_max,
    feh_min,
    feh_max,
    iso_surface,
    order,
):
    model = DifferencePolyFehMassAbsgModel(
        order=order,
        absg_min=absg_min,
        absg_max=absg_max,
        mass_min=mass_min,
        pivot=(absg_min + absg_max) / 2.0,
        deriv_penalty_strength=0.0,
        isochrone_surface_model=iso_surface,
        feh_min=feh_min,
        feh_max=feh_max,
    )

    absg_range = np.linspace(absg_min, absg_max, 600)
    feh_values = np.asarray(feh_values, dtype=float)
    feh_values = np.sort(feh_values[np.isfinite(feh_values)])

    cmap = plt.cm.viridis
    norm = plt.Normalize(feh_values.min(), feh_values.max())

    fig, ax = plt.subplots(figsize=(10, 6))

    for feh in feh_values:
        color = cmap(norm(feh))
        if params_samples is not None:
            masses = model.mass_from_absg_feh(absg_range, feh, params_samples)
            lower, median, upper = np.percentile(masses, [16, 50, 84], axis=0)
            ax.fill_between(absg_range, lower, upper, color=color, alpha=0.18)
            ax.plot(absg_range, median, color=color, linewidth=2)
        else:
            masses = model.mass_from_absg_feh(absg_range, feh, params)
            ax.plot(absg_range, masses, color=color, linewidth=2)

        iso_mass = iso_surface.mass_from_absg_mh(absg_range, feh)
        ax.plot(absg_range, iso_mass, color=color, linewidth=1.5, linestyle="--", alpha=0.8)

    ax.set_xlabel("$M_{\\mathrm{G}}$ [mag]")
    ax.set_ylabel("Mass [$M_{\\odot}$]")
    ax.set_xlim(absg_min, absg_max)
    ax.set_ylim(mass_min * 0.8, mass_max * 1.2)
    ax.set_yscale("log")
    ax.invert_xaxis()

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, pad=0.02)
    cbar.set_label("[Fe/H]", rotation=270, labelpad=12)

    ax.legend(["Fit", "Isochrone"], loc="best")
    plt.tight_layout()
    plt.show()


In [ ]:
# Run plot
plot_mlr_multi_feh(
    params=params,
    params_samples=params_samples,
    feh_values=feh_values,
    absg_min=absg_min,
    absg_max=absg_max,
    mass_min=mass_min,
    mass_max=mass_max,
    feh_min=feh_min,
    feh_max=feh_max,
    iso_surface=iso_surface,
    order=order,
)
